# `wc_cosmict_2_dQ_dx_end` Data/MC Study

`wc_cosmict_2_dQ_dx_end` is the dQ/dx near the end of the main track examined by Wire-Cell cosmic tagger 2, in MIP units.
In the BDT-variable p-value scan it is one of the most discrepant variables (χ² = 62.6/22 with DetVar).
Data has a deficit around 1.5–1.8 and an excess around 2.0–2.4, which looks like the peak at the track end sitting at higher dQ/dx in data.
`wc_cosmict_2_dQ_dx_front`, on the same tracks, agrees. The rate of events with cosmic tagger 2 filled also agrees (`wc_cosmict_4_filled`, χ² = 0.04/2).
The DetVar error in the excess region is ~90%, almost all from the Recomb2 variation.

This notebook looks for where the shift comes from:
1. **Composition**: which event types populate the excess region.
2. **EXT vs overlay cosmics**: both are real detector data (overlay cosmics are data waveforms with simulated neutrinos added),
   so their shapes should match unless the overlay and data processing or calibration differ. Also beam-on minus beam-off data vs the overlay prediction.
3. **Splits**: the excess-region fraction, data vs prediction, by run period, track angle, particle type, containment and direction flags.
4. **DetVar**: how each detector variation moves the distribution, and whether any variation reproduces the data/MC shift.

Everything uses statistical uncertainties only, apart from the DetVar section. The question is where the *shape* differs, and shape fractions cancel the
normalization systematics. Event selection is the preselection with cosmic tagger 2 filled.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from src.file_locations import intermediate_files_location
from src.df_helpers import DETVAR_WEIGHT_COL, DETVAR_MATCH_KEYS, detvar_weight_col

plt.rcParams.update({"font.size": 12})
pd.set_option("display.width", 200)
out_dir = Path("../plots/cosmict_2_dQ_dx_end_study")
out_dir.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(out_dir / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(out_dir / f"{name}.png", bbox_inches="tight", dpi=150)

## Load events

In [ ]:
cols = [
    "wc_cosmict_2_filled", "wc_cosmict_7_filled",
    "wc_cosmict_2_dQ_dx_end", "wc_cosmict_2_dQ_dx_front", "wc_cosmict_2_theta", "wc_cosmict_2_phi", "wc_cosmict_2_angle_beam",
    "wc_cosmict_2_particle_type", "wc_cosmict_2_flag_inside", "wc_cosmict_2_flag_dir_weak", "wc_cosmict_2_n_muon_tracks",
    "filetype", "run", "normalizing_run_period_open_data", "wc_net_weight_open_data",
    "wc_match_completeness_energy", "wc_truth_energyInside", "wc_truth_inFV", "wc_truth_numuCC",
]
# same open-data prediction samples as plot_many_variables.py
pred_filetypes = ["nu_overlay", "nue_overlay", "dirt_overlay", "ext", "nc_pi0_overlay", "numucc_pi0_overlay",
                  "ncdeltarad_overlay", "NC_coherent_1g_reweighted", "numucc_rad_corr_sim"]
df = pl.read_parquet(f"{intermediate_files_location}/presel_df_train_vars.parquet", columns=cols)
df = df.filter((pl.col("wc_cosmict_2_filled") == 1) & pl.col("filetype").is_in(pred_filetypes + ["data"]))

# origin of each prediction event; overlay events whose reconstructed activity is <10% matched to the
# true neutrino energy deposit are "cosmic-dominated" (the Wire-Cell cosmic definition), so the tagged track is a data cosmic
cosmic_dominated = (pl.col("wc_truth_energyInside") <= 0) | (pl.col("wc_match_completeness_energy") < 0.1 * pl.col("wc_truth_energyInside"))
df = df.with_columns(
    pl.when(pl.col("filetype") == "data").then(pl.lit("data"))
    .when(pl.col("filetype") == "ext").then(pl.lit("EXT (beam-off data)"))
    .when(cosmic_dominated).then(pl.lit("overlay, cosmic-dominated"))
    .when(pl.col("filetype") == "dirt_overlay").then(pl.lit("dirt"))
    .when(~pl.col("wc_truth_inFV")).then(pl.lit("ν out-FV"))
    .when(pl.col("wc_truth_numuCC")).then(pl.lit("νμ CC in-FV"))
    .otherwise(pl.lit("other ν in-FV"))
    .alias("origin"),
    pl.col("wc_cosmict_2_theta").alias("theta"),
)
origins = ["νμ CC in-FV", "other ν in-FV", "ν out-FV", "dirt", "overlay, cosmic-dominated", "EXT (beam-off data)"]
origin_colors = dict(zip(origins, ["xkcd:sky blue", "xkcd:light orange", "xkcd:lavender", "xkcd:brown", "xkcd:grey", "xkcd:green"]))

data = df.filter(pl.col("origin") == "data")
pred = df.filter(pl.col("origin") != "data")
print(f"data events: {data.height},  prediction: {pred.get_column('wc_net_weight_open_data').sum():.1f} weighted ({pred.height} MC events)")
print(pred.group_by("origin").agg(pl.col("wc_net_weight_open_data").sum().alias("weighted"), pl.len().alias("mc_events"))
      .with_columns((pl.col("weighted") / pl.col("weighted").sum()).alias("fraction")).sort("weighted", descending=True))

In [ ]:
# fixed binning over the core of the distribution; the long tails (down to ~-100, up to ~1e7) are counted separately
bins = np.round(np.arange(0.5, 3.5001, 0.1), 2)
centers = 0.5 * (bins[1:] + bins[:-1])
DIP = (1.5, 1.8)
BUMP = (2.0, 2.4)
CORE = (0.8, 3.2)

def vals(d, var="wc_cosmict_2_dQ_dx_end"):
    return d.get_column(var).to_numpy().astype(float)

def w(d):
    return d.get_column("wc_net_weight_open_data").to_numpy().astype(float) if d.height and d.get_column("filetype")[0] != "data" else np.ones(d.height)

def hist(d, var="wc_cosmict_2_dQ_dx_end", weights=None):
    x = vals(d, var)
    ww = w(d) if weights is None else weights
    h = np.histogram(x, bins=bins, weights=ww)[0]
    h2 = np.histogram(x, bins=bins, weights=ww**2)[0]
    return h, h2

for name, d in [("data", data), ("prediction", pred)]:
    x, ww = vals(d), w(d)
    print(f"{name:10s} below {bins[0]}: {ww[x < bins[0]].sum() / ww.sum():.2%},  above {bins[-1]}: {ww[x > bins[-1]].sum() / ww.sum():.2%}")

## 1. Composition

Stacked prediction by origin, with data and the data/prediction ratio (stat errors only). The dQ/dx_front panel on the same events is a control.

In [ ]:
def stack_plot(var, name, xlabel):
    fig, (ax, axr) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})
    bottom = np.zeros(len(centers))
    tot = np.zeros(len(centers)); tot2 = np.zeros(len(centers))
    for o in origins:
        h, h2 = hist(pred.filter(pl.col("origin") == o), var)
        ax.bar(centers, h, width=np.diff(bins), bottom=bottom, color=origin_colors[o], label=o, align="center")
        bottom += h; tot += h; tot2 += h2
    hd, _ = hist(data, var)
    ax.errorbar(centers, hd, yerr=np.sqrt(hd), fmt="o", color="k", ms=3, label="data")
    ax.set_ylabel("events"); ax.legend(fontsize=9)
    r = np.divide(hd, tot, out=np.ones_like(hd), where=tot > 0)
    rerr = np.divide(np.sqrt(hd), tot, out=np.zeros_like(hd), where=tot > 0)
    axr.errorbar(centers, r, yerr=rerr, fmt="o", color="k", ms=3)
    rel = np.divide(np.sqrt(tot2), tot, out=np.zeros_like(tot), where=tot > 0)
    axr.fill_between(bins, np.r_[1 - rel, 1 - rel[-1]], np.r_[1 + rel, 1 + rel[-1]], step="post", color="grey", alpha=0.3, label="MC stat")
    for lo, hi, c in [(*DIP, "C0"), (*BUMP, "C3")]:
        for a in (ax, axr):
            a.axvspan(lo, hi, color=c, alpha=0.08)
    axr.axhline(1, color="k", lw=0.5); axr.set_ylim(0.5, 1.8); axr.set_ylabel("data/pred"); axr.set_xlabel(xlabel)
    save(fig, name)
    plt.show()

stack_plot("wc_cosmict_2_dQ_dx_end", "composition_dQ_dx_end", "wc_cosmict_2_dQ_dx_end (shaded: dip and excess windows)")
stack_plot("wc_cosmict_2_dQ_dx_front", "composition_dQ_dx_front", "wc_cosmict_2_dQ_dx_front")

In [ ]:
def window_frac(d, lo, hi, var="wc_cosmict_2_dQ_dx_end"):
    # weighted fraction of the CORE events that fall in [lo, hi), with its binomial-style stat error
    x, ww = vals(d, var), w(d)
    core = (x >= CORE[0]) & (x < CORE[1])
    inw = core & (x >= lo) & (x < hi)
    W, Win = ww[core].sum(), ww[inw].sum()
    if W <= 0:
        return np.nan, np.nan
    f = Win / W
    # variance of a ratio of weighted sums (in and out of the window are independent)
    var_in, var_out = (ww[inw] ** 2).sum(), (ww[core & ~inw] ** 2).sum()
    err = np.sqrt((1 - f) ** 2 * var_in + f ** 2 * var_out) / W
    return f, err

rows = [dict(sample="data", **{f"{k}_frac": window_frac(data, *win)[0] for k, win in [("dip", DIP), ("excess", BUMP)]})]
rows += [dict(sample="prediction (all)", **{f"{k}_frac": window_frac(pred, *win)[0] for k, win in [("dip", DIP), ("excess", BUMP)]})]
rows += [dict(sample=o, **{f"{k}_frac": window_frac(pred.filter(pl.col("origin") == o), *win)[0] for k, win in [("dip", DIP), ("excess", BUMP)]}) for o in origins]
print(f"fraction of events in [{CORE[0]}, {CORE[1]}) that are in the dip {DIP} and excess {BUMP} windows")
pd.DataFrame(rows).round(4)

## 2. EXT vs overlay cosmics, and beam-on minus beam-off

Area-normalized shapes in the core range. EXT and the cosmic-dominated overlay events both have a real-data cosmic as the tagged track,
so a shape difference between them points to the overlay processing or calibration rather than to simulation physics.
Beam-on data minus EXT (beam-off) is compared with the non-EXT prediction. If the shift is still there, it is in the beam-related events.

In [ ]:
def norm_shape(h, h2):
    core = (centers >= CORE[0]) & (centers < CORE[1])
    s = h[core].sum()
    return h / s, np.sqrt(np.maximum(h2, 0)) / s

hd, hd2 = hist(data)
he, he2 = hist(pred.filter(pl.col("origin") == "EXT (beam-off data)"))
hc, hc2 = hist(pred.filter(pl.col("origin") == "overlay, cosmic-dominated"))
hn, hn2 = hist(pred.filter(~pl.col("origin").is_in(["EXT (beam-off data)", "overlay, cosmic-dominated"])))
hmc, hmc2 = hist(pred.filter(pl.col("origin") != "EXT (beam-off data)"))
hon, hon2 = hd - he, hd + he2   # beam-on minus beam-off; EXT carries its beam-off -> beam-on scale in its weight

fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})
for col, (title, a, b, la, lb) in enumerate([
        ("cosmic tracks: EXT vs overlay", (he, he2), (hc, hc2), "EXT (beam-off data)", "overlay, cosmic-dominated"),
        ("beam-related: data − EXT vs non-EXT prediction", (hon, hon2), (hmc, hmc2), "data − EXT", "prediction without EXT")]):
    ax, axr = axes[0, col], axes[1, col]
    sa, ea = norm_shape(*a); sb, eb = norm_shape(*b)
    ax.errorbar(centers, sa, yerr=ea, fmt="o", ms=3, color="k", label=la)
    ax.step(bins, np.r_[sb, sb[-1]], where="post", color="C3", label=lb)
    ax.set_title(title); ax.legend(fontsize=9); ax.set_ylabel("area-normalized (core range)")
    r = np.divide(sa, sb, out=np.ones_like(sa), where=sb > 0)
    re = np.divide(ea, sb, out=np.zeros_like(sa), where=sb > 0)
    axr.errorbar(centers, r, yerr=re, fmt="o", ms=3, color="k")
    axr.axhline(1, color="k", lw=0.5); axr.set_ylim(0.4, 2.0); axr.set_ylabel(f"ratio"); axr.set_xlabel("wc_cosmict_2_dQ_dx_end")
    for lo, hi, c in [(*DIP, "C0"), (*BUMP, "C3")]:
        for a_ in (ax, axr):
            a_.axvspan(lo, hi, color=c, alpha=0.08)
save(fig, "shapes_ext_vs_overlay_cosmics")
plt.show()

## 3. Where is the shift? Splits of the excess-window fraction

For each split, the fraction of core-range events in the excess window (2.0–2.4) for data and prediction, and their ratio.
The ratio is about 1.3 overall. A split where it is near 1 is a part of phase space without the problem.

In [ ]:
def split_table(expr_or_col, labels=None):
    rows = []
    groups = data.get_column(expr_or_col).unique().sort().to_list() if isinstance(expr_or_col, str) else None
    for g in groups:
        dsub = data.filter(pl.col(expr_or_col) == g); psub = pred.filter(pl.col(expr_or_col) == g)
        fd, ed = window_frac(dsub, *BUMP); fp, ep = window_frac(psub, *BUMP)
        rows.append(dict(split=labels.get(g, g) if labels else g, data_events=dsub.height,
                         pred_events=psub.get_column("wc_net_weight_open_data").sum(),
                         excess_frac_data=fd, excess_frac_pred=fp, ratio=fd / fp, ratio_err=np.hypot(ed / fp, fd * ep / fp ** 2)))
    return pd.DataFrame(rows)

def plot_split(table, title, name):
    fig, ax = plt.subplots(figsize=(max(5, 0.9 * len(table) + 2), 3.5))
    x = np.arange(len(table))
    ax.errorbar(x, table.ratio, yerr=table.ratio_err, fmt="o", color="k")
    ax.axhline(1, color="k", lw=0.5)
    ax.set_xticks(x, [str(s) for s in table.split], rotation=30, ha="right")
    ax.set_ylabel("excess-window fraction\ndata / pred"); ax.set_title(title)
    save(fig, name)
    plt.show()

# binned versions of the continuous split variables. Angles are in degrees; wc_cosmict_2_theta is identical to
# wc_cosmict_2_angle_beam (polar angle from the beam, z), and phi is the azimuth atan2(y, x).
# The angle to the drift direction (x) matters for recombination: cos(angle to x) = sin(theta) cos(phi)
def binned(expr, edges, name):
    labels = [f"{a:+04.0f} to {b:+04.0f}" for a, b in zip(edges[:-1], edges[1:])]  # sortable as text
    return expr.cut(list(edges[1:-1]), labels=labels).cast(pl.Utf8).alias(name)

def add_bins(d):
    th, ph = np.radians(1) * pl.col("wc_cosmict_2_theta"), np.radians(1) * pl.col("wc_cosmict_2_phi")
    angle_to_drift = ((th.sin() * ph.cos()).abs().clip(0, 1).arccos() * 180 / np.pi)
    return d.with_columns(
        binned(pl.col("wc_cosmict_2_angle_beam"), np.arange(0, 181, 30), "angle_beam_bin"),
        binned(pl.col("wc_cosmict_2_phi"), np.arange(-180, 181, 60), "phi_bin"),
        binned(angle_to_drift, np.arange(0, 91, 15), "angle_to_drift_bin"),
    )
data, pred = add_bins(data), add_bins(pred)

In [ ]:
splits = [
    ("normalizing_run_period_open_data", "run period"),
    ("wc_cosmict_7_filled", "cosmic tagger 7 also filled"),
    ("wc_cosmict_2_particle_type", "particle type of the tagged track"),
    ("wc_cosmict_2_flag_inside", "track end inside (flag_inside)"),
    ("wc_cosmict_2_flag_dir_weak", "weak direction (flag_dir_weak)"),
    ("wc_cosmict_2_n_muon_tracks", "number of muon tracks"),
    ("angle_beam_bin", "track angle to the beam (deg)"),
    ("phi_bin", "track azimuth φ (deg)"),
    ("angle_to_drift_bin", "track angle to the drift direction x (deg; 90 = parallel to the wire planes)"),
]
split_tables = {}
for col, title in splits:
    t = split_table(col)
    t = t[(t.data_events >= 200)]  # drop tiny groups
    split_tables[col] = t
    print(f"\n=== {title} ===")
    print(t.round(4).to_string(index=False))
    plot_split(t, title, f"split_{col}")

## 3a. Composition in regions of track angle to the drift direction

The same stacked histogram as section 1, split by the tagged track's angle to the drift direction x
(0° = along the drift, 90° = parallel to the wire planes). The data/prediction normalization in each panel is free,
because the panel's rate isn't the question, so the ratio shown is shape-only: the prediction is scaled to the data count in the panel's core range.

In [ ]:
def stack_on_axes(ax, axr, d_data, d_pred, var="wc_cosmict_2_dQ_dx_end", shape_only=True):
    core = (centers >= CORE[0]) & (centers < CORE[1])
    hd, _ = hist(d_data, var)
    hs = {o: hist(d_pred.filter(pl.col("origin") == o), var) for o in origins}
    tot = sum(h for h, _ in hs.values()); tot2 = sum(h2 for _, h2 in hs.values())
    scale = hd[core].sum() / tot[core].sum() if shape_only else 1.0
    bottom = np.zeros(len(centers))
    for o in origins:
        h = hs[o][0] * scale
        ax.bar(centers, h, width=np.diff(bins), bottom=bottom, color=origin_colors[o], label=o, align="center")
        bottom += h
    tot, tot2 = tot * scale, tot2 * scale ** 2
    ax.errorbar(centers, hd, yerr=np.sqrt(hd), fmt="o", color="k", ms=2.5, label="data")
    r = np.divide(hd, tot, out=np.ones_like(hd), where=tot > 0)
    rerr = np.divide(np.sqrt(hd), tot, out=np.zeros_like(hd), where=tot > 0)
    rel = np.divide(np.sqrt(tot2), tot, out=np.zeros_like(tot), where=tot > 0)
    axr.fill_between(bins, np.r_[1 - rel, 1 - rel[-1]], np.r_[1 + rel, 1 + rel[-1]], step="post", color="grey", alpha=0.3)
    axr.errorbar(centers, r, yerr=rerr, fmt="o", color="k", ms=2.5)
    axr.axhline(1, color="k", lw=0.5); axr.set_ylim(0.3, 2.2)
    for lo, hi, c in [(*DIP, "C0"), (*BUMP, "C3")]:
        for a in (ax, axr):
            a.axvspan(lo, hi, color=c, alpha=0.08)
    return scale

th, ph = np.radians(1) * pl.col("wc_cosmict_2_theta"), np.radians(1) * pl.col("wc_cosmict_2_phi")
angle_to_drift = ((th.sin() * ph.cos()).abs().clip(0, 1).arccos() * 180 / np.pi).alias("angle_to_drift")
data, pred = data.with_columns(angle_to_drift), pred.with_columns(angle_to_drift)

def angle_panels(region_edges, name, extra_cut=None, extra_cut_text=None):
    regions = list(zip(region_edges[:-1], region_edges[1:]))
    fig, axes = plt.subplots(2, len(regions), figsize=(4.6 * len(regions), 7.2), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05, "wspace": 0.25})
    # selection common to all panels, and the definition of the angle the panels are split in
    common = "Preselection, open data, wc_cosmict_2_filled == 1"
    if extra_cut_text:
        common += ", " + extra_cut_text
    fig.suptitle(common + "\n"
                 "Panels: α = angle between the cosmic-tagger-2 track and the drift direction x = "
                 "arccos(|sin θ · cos φ|),  θ = wc_cosmict_2_theta (deg from beam),  φ = wc_cosmict_2_phi (deg)\n"
                 "α = 0°: track along the drift;  α = 90°: track parallel to the wire planes",
                 fontsize=11, y=1.04)
    for k, (lo, hi) in enumerate(regions):
        sel = (pl.col("angle_to_drift") >= lo) & (pl.col("angle_to_drift") < hi)
        if extra_cut is not None:
            sel = sel & extra_cut
        d_data, d_pred = data.filter(sel), pred.filter(sel)
        scale = stack_on_axes(axes[0, k], axes[1, k], d_data, d_pred)
        fd, ed = window_frac(d_data, *BUMP); fp, ep = window_frac(d_pred, *BUMP)
        axes[0, k].set_title(f"{lo:.0f}° ≤ α < {hi:.0f}°\n{d_data.height} data events, pred × {scale:.2f}\n"
                             f"excess-window frac. data/pred = {fd / fp:.2f} ± {np.hypot(ed / fp, fd * ep / fp ** 2):.2f}", fontsize=10)
        axes[1, k].set_xlabel("wc_cosmict_2_dQ_dx_end")
    axes[0, 0].set_ylabel("events"); axes[1, 0].set_ylabel("data / pred\n(shape only)")
    axes[0, -1].legend(fontsize=7, loc="upper right")
    save(fig, name)
    plt.show()

angle_edges = [0, 30, 50, 70, 90]
angle_panels(angle_edges, "composition_by_angle_to_drift")

Same panels for stopping muon-like tracks only (particle_type 13, track end inside), which make up the second peak.

In [ ]:
angle_panels(angle_edges, "composition_by_angle_to_drift_stopping_muons",
             extra_cut=(pl.col("wc_cosmict_2_particle_type") == 13) & (pl.col("wc_cosmict_2_flag_inside") == 1),
             extra_cut_text="stopping muon-like: wc_cosmict_2_particle_type == 13 and wc_cosmict_2_flag_inside == 1")

## 3b. Position of the stopping-muon peak

Events where the tagged track is muon-like (`particle_type` 13) and ends inside the detector (`flag_inside` 1).
These are the events behind the second peak near dQ/dx ≈ 2.
Below is the weighted median of `dQ_dx_end` in [1.4, 2.8], where the second peak is, for each run period.

In [ ]:
PEAK = (1.4, 2.8)
def wmedian(d):
    x, ww = vals(d), w(d)
    m = (x >= PEAK[0]) & (x < PEAK[1])
    if m.sum() < 20:
        return np.nan
    o = np.argsort(x[m]); cw = np.cumsum(ww[m][o])
    return x[m][o][np.searchsorted(cw, cw[-1] / 2)]

stopping = (pl.col("wc_cosmict_2_particle_type") == 13) & (pl.col("wc_cosmict_2_flag_inside") == 1)
samples = {
    "data": data.filter(stopping),
    "prediction (all)": pred.filter(stopping),
    "νμ CC in-FV": pred.filter(stopping & (pl.col("origin") == "νμ CC in-FV")),
    "EXT (beam-off data)": pred.filter(stopping & (pl.col("origin") == "EXT (beam-off data)")),
    "overlay, cosmic-dominated": pred.filter(stopping & (pl.col("origin") == "overlay, cosmic-dominated")),
}
periods = sorted(data.get_column("normalizing_run_period_open_data").unique().to_list())
peak_rows = []
for name, d in samples.items():
    row = {"sample": name, "all periods": wmedian(d)}
    for per in periods:
        row[f"run {per}"] = wmedian(d.filter(pl.col("normalizing_run_period_open_data") == per))
    peak_rows.append(row)
peak_table = pd.DataFrame(peak_rows).set_index("sample")
print(f"weighted median of wc_cosmict_2_dQ_dx_end in {PEAK}, stopping muon-like tracks")
peak_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, color, style in [("data", "k", "-"), ("prediction (all)", "C3", "--"), ("EXT (beam-off data)", "xkcd:green", "-"), ("overlay, cosmic-dominated", "xkcd:grey", "--")]:
    d = samples[name]
    h = np.histogram(vals(d), bins=bins, weights=w(d))[0]
    ax.step(bins, np.r_[h, h[-1]] / h.sum(), where="post", color=color, ls=style, lw=1.5, label=name)
ax.set_xlabel("wc_cosmict_2_dQ_dx_end"); ax.set_ylabel("area-normalized")
ax.set_title("stopping muon-like tracks (particle_type 13, flag_inside 1)")
ax.legend()
save(fig, "stopping_muon_shapes")
plt.show()

## 4. Detector variations

Each DetVar sample is compared with its own matched CV events, as in the DetVar covariance.
Each variation's fractional change in the dip and excess windows is compared with the data/prediction difference,
to see if one variation reproduces the data shape.

In [ ]:
detvar_cols = ["vartype", "filetype", "run", "subrun", "event", DETVAR_WEIGHT_COL, "wc_net_weight_sce_recomb2",
               "wc_cosmict_2_filled", "wc_cosmict_2_dQ_dx_end"]
detvar = pl.read_parquet(f"{intermediate_files_location}/detvar_presel_df_train_vars.parquet", columns=detvar_cols)
detvar = detvar.filter(pl.col("wc_cosmict_2_filled") == 1)
print(detvar.group_by("vartype").len().sort("vartype"))

In [ ]:
vartypes = ["LYAtt", "LYDown", "LYRayleigh", "WireModX", "WireModYZ", "WireModThetaXZ", "WireModThetaYZ", "Recomb2", "SCE"]
cv = detvar.filter(pl.col("vartype") == "CV")
rows, shapes = [], {}
for vt in vartypes:
    v = detvar.filter(pl.col("vartype") == vt)
    if v.height == 0:
        continue
    wcol = detvar_weight_col(vt)
    cv_group = cv.filter(pl.col(wcol).is_not_null())
    # events present in both the CV and this variation (each may be selected in only one of them)
    keys = cv_group.select(DETVAR_MATCH_KEYS).unique().join(v.select(DETVAR_MATCH_KEYS).unique(), on=DETVAR_MATCH_KEYS, how="inner")
    cv_m = cv_group.join(keys, on=DETVAR_MATCH_KEYS, how="semi")
    v_m = v.join(keys, on=DETVAR_MATCH_KEYS, how="semi")
    hc_ = np.histogram(vals(cv_m), bins=bins, weights=cv_m.get_column(wcol).to_numpy())[0]
    hv_ = np.histogram(vals(v_m), bins=bins, weights=v_m.get_column(wcol).to_numpy())[0]
    shapes[vt] = (hc_, hv_)
    core = (centers >= CORE[0]) & (centers < CORE[1])
    def frac(h, lo, hi):
        m = core & (centers >= lo) & (centers < hi)
        return h[m].sum() / h[core].sum()
    rows.append(dict(vartype=vt, matched_cv_events=cv_m.height,
                     dip_frac_change=frac(hv_, *DIP) / frac(hc_, *DIP) - 1,
                     excess_frac_change=frac(hv_, *BUMP) / frac(hc_, *BUMP) - 1,
                     median_shift=np.median(vals(v_m)[(vals(v_m) > CORE[0]) & (vals(v_m) < CORE[1])]) -
                                  np.median(vals(cv_m)[(vals(cv_m) > CORE[0]) & (vals(cv_m) < CORE[1])])))
detvar_table = pd.DataFrame(rows)

def med(d):
    x = vals(d); ww = w(d)
    m = (x > CORE[0]) & (x < CORE[1])
    o = np.argsort(x[m]); cw = np.cumsum(ww[m][o])
    return x[m][o][np.searchsorted(cw, cw[-1] / 2)]
fd_dip, _ = window_frac(data, *DIP); fp_dip, _ = window_frac(pred, *DIP)
fd_ex, _ = window_frac(data, *BUMP); fp_ex, _ = window_frac(pred, *BUMP)
data_row = dict(vartype="DATA / prediction", matched_cv_events=data.height, dip_frac_change=fd_dip / fp_dip - 1,
                excess_frac_change=fd_ex / fp_ex - 1, median_shift=med(data) - med(pred))
pd.concat([detvar_table, pd.DataFrame([data_row])]).round(4)

In [ ]:
fig, (ax, axr) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1], "hspace": 0.05})
core = (centers >= CORE[0]) & (centers < CORE[1])
hd_, _ = hist(data); hp_, _ = hist(pred)
ax.step(bins, np.r_[hd_ / hd_[core].sum(), 0], where="post", color="k", lw=2, label="data")
ax.step(bins, np.r_[hp_ / hp_[core].sum(), 0], where="post", color="k", ls="--", label="prediction")
r_data = (hd_ / hd_[core].sum()) / (hp_ / hp_[core].sum())
axr.step(bins, np.r_[r_data, r_data[-1]], where="post", color="k", lw=2, label="data / pred")
for i, vt in enumerate(vartypes):
    if vt not in shapes:
        continue
    hc_, hv_ = shapes[vt]
    r = (hv_ / hv_[core].sum()) / (hc_ / hc_[core].sum())
    lw = 2 if vt in ("Recomb2", "SCE") else 1
    axr.step(bins, np.r_[r, r[-1]], where="post", color=f"C{i}", lw=lw, label=f"{vt} / CV")
ax.set_ylabel("area-normalized"); ax.legend(fontsize=9)
axr.axhline(1, color="k", lw=0.5); axr.set_ylim(0.3, 2.2); axr.set_ylabel("shape ratio"); axr.set_xlabel("wc_cosmict_2_dQ_dx_end")
axr.legend(fontsize=7, ncol=3, loc="upper left")
save(fig, "detvar_shape_ratios")
plt.show()

## 5. Angle slices with full systematics

The four α regions of section 3a, joined end to end into one histogram ("2D slices"): slice k holds `wc_cosmict_2_dQ_dx_end`
for events with α in region k, using the same dQ/dx bins in every slice. Values outside the slice range go into its first or last bin.
This goes through `make_histogram_plot` with the full reweightable systematics (flux, GENIE, reinteraction), MC and data statistics, and the DetVar covariance,
so both χ²/ndf values (without and with DetVar) include the correlations between slices.
The covariance cache keys use a dedicated selname per selection, so they don't collide with the preselection caches.

In [ ]:
from src.plot_helpers import make_histogram_plot
from src.signal_categories import del1g_detailed_category_queries

presel_path = f"{intermediate_files_location}/presel_df_train_vars.parquet"
detvar_path = f"{intermediate_files_location}/detvar_presel_df_train_vars.parquet"
presel_schema = pl.read_parquet_schema(presel_path)
detvar_schema = pl.read_parquet_schema(detvar_path)

# the same base columns plot_many_variables.py loads, plus the cosmic-tagger-2 columns used here
category_columns = set()
for query in del1g_detailed_category_queries:
    category_columns |= set(eval(query, {"pl": pl, "__builtins__": {}}).meta.root_names())
base_columns = (
    {"filename", "run", "subrun", "event", "filetype", "vartype",
     "wc_net_weight_open_data", "wc_net_weight", "wc_net_weight_sce_recomb2", "pandora_weightTune",
     "normal_overlay", "del1g_overlay", "iso1g_overlay",
     "wc_cosmict_2_filled", "wc_cosmict_2_theta", "wc_cosmict_2_phi", "wc_cosmict_2_dQ_dx_end",
     "wc_cosmict_2_particle_type", "wc_cosmict_2_flag_inside",
     "wc_truth_muonMomentum_0", "wc_truth_muonMomentum_1", "wc_truth_muonMomentum_2", "wc_truth_muonMomentum_3",
     "wc_true_max_prim_proton_costheta", "wc_true_max_prim_proton_phi"}
    | {c for c in list(presel_schema) + list(detvar_schema) if c.startswith(("norm_goal_pot_", "normalizing_run_period_"))}
    | category_columns
)

SLICE_EDGES = [0, 30, 50, 70, 90]             # α regions, degrees
SLICE_LO, SLICE_HI, SLICE_STEP = 0.6, 3.0, 0.2  # dQ/dx bins within each slice
SLICE_WIDTH = SLICE_HI - SLICE_LO
SLICED_VAR = "cosmict_2_dQ_dx_end_angle_slices"
n_slices = len(SLICE_EDGES) - 1
within = np.round(np.arange(SLICE_LO, SLICE_HI + 1e-9, SLICE_STEP), 6)
slice_bins = np.unique(np.round(np.concatenate([within + k * SLICE_WIDTH for k in range(n_slices)]), 6))

def add_sliced_var(d):
    th, ph = np.radians(1) * pl.col("wc_cosmict_2_theta"), np.radians(1) * pl.col("wc_cosmict_2_phi")
    alpha = (th.sin() * ph.cos()).abs().clip(0, 1).arccos() * 180 / np.pi
    k = pl.lit(None, dtype=pl.Float64)
    for i, (lo, hi) in enumerate(zip(SLICE_EDGES[:-1], SLICE_EDGES[1:])):
        upper = (alpha <= hi) if i == n_slices - 1 else (alpha < hi)
        k = pl.when((alpha >= lo) & upper).then(pl.lit(float(i))).otherwise(k)
    x = pl.col("wc_cosmict_2_dQ_dx_end").cast(pl.Float64).clip(SLICE_LO, SLICE_HI - 1e-6)
    # only events with cosmic tagger 2 filled and a defined angle get a value; the rest are dropped below
    return d.with_columns(pl.when(pl.col("wc_cosmict_2_filled") == 1).then(x + k * SLICE_WIDTH).otherwise(None).alias(SLICED_VAR))

def read_base(path, schema):
    return pl.read_parquet(path, columns=[c for c in schema if c in base_columns])

pred_filter = pl.col("filetype").is_in(pred_filetypes)
presel_all_df = add_sliced_var(read_base(presel_path, presel_schema))
detvar_all_df = add_sliced_var(read_base(detvar_path, detvar_schema))
full_df = presel_all_df.filter(pl.col(SLICED_VAR).is_not_null())
full_detvar_df = detvar_all_df.filter(pl.col(SLICED_VAR).is_not_null())
weights_df = pl.read_parquet(f"{intermediate_files_location}/presel_weights_df.parquet")
print(full_df.shape, full_detvar_df.shape, weights_df.shape)

In [ ]:
last_recomb2_components = None  # filled by recomb2_decorrelated_dic (section 6): CV and per-region Recomb2 - CV histograms

def add_recomb2_panel(fig, display_bins):
    # extra column right of the DetVar variation panels: the Recomb2 group's matched CV, the full Recomb2 spectrum,
    # and CV + the shift of each decorrelated piece (only that true-angle region varied), with (Var - CV)/CV below
    comps = last_recomb2_components
    ax_vt = next(a for a in fig.axes if a.get_ylabel().startswith("MC Pred counts"))
    ax_vb = next(a for a in fig.axes if a.get_ylabel().startswith("(Var"))
    ax_sb = next(a for a in fig.axes if a.get_title() == "Systematic Breakdown")
    dx = ax_vt.get_position().x0 - ax_sb.get_position().x0
    new_axes = []
    for src in (ax_vt, ax_vb):
        pos = src.get_position()
        new_axes.append(fig.add_axes([pos.x0 + dx, pos.y0, pos.width, pos.height]))
    ax_t, ax_b = new_axes
    cv = comps["cv"]
    edges = np.asarray(display_bins, dtype=float)
    def step(ax, y, **kw):
        ax.step(edges, np.r_[y, y[-1]], where="post", **kw)
    step(ax_t, cv, color="k", lw=2, label="CV (Recomb2 group, matched)")
    step(ax_t, cv + comps["total_diff"], color="C7", lw=2, label="Recomb2 (all regions)")
    frac = lambda d: np.divide(d, cv, out=np.zeros_like(d, dtype=float), where=cv > 0)
    step(ax_b, frac(comps["total_diff"]), color="C7", lw=2, label="Recomb2 (all regions)")
    for i, (label, d) in enumerate(comps["pieces"].items()):
        color = ["C4", "C0", "C2", "C1", "C3"][i % 5]
        step(ax_t, cv + d, color=color, lw=1, label=f"CV + {label} shift")
        step(ax_b, frac(d), color=color, lw=1)
    ax_t.set_yscale(ax_vt.get_yscale())
    ax_t.set_xlim(ax_vt.get_xlim()); ax_b.set_xlim(ax_vb.get_xlim())
    ax_t.set_xticklabels([])
    ax_b.set_ylim(-1, 1); ax_b.axhline(0, color="k", lw=0.5)
    ax_t.set_title("Recomb2: total and decorrelated pieces")
    ax_t.set_ylabel("MC counts, Recomb2 CV group"); ax_b.set_ylabel("(Var − CV)/CV")
    ax_b.set_xlabel(ax_vb.get_xlabel())
    ax_t.legend(fontsize=8)
    return ax_t, ax_b

def sliced_plot(cut, cut_text, selname, name, detvar_override=None):
    d = full_df if cut is None else full_df.filter(cut)
    dv = full_detvar_df if cut is None else full_detvar_df.filter(cut)
    info = make_histogram_plot(
        pred_sel_df=d.filter(pred_filter), data_sel_df=d.filter(pl.col("filetype") == "data"),
        bins=slice_bins, include_overflow=False, include_underflow=False, log_y=False, include_legend=False,
        var=SLICED_VAR, display_var="wc_cosmict_2_dQ_dx_end, in slices of α (angle to drift)",
        title=f"Preselection, wc_cosmict_2_filled == 1{cut_text}", selname=selname,
        include_ratio=True, include_decomposition=True, include_sys_breakdown_panels=True, include_detvar_variation_panels=True,
        use_rw_systematics=True, use_detvar_systematics=True, detvar_df=dv, weights_df=weights_df, require_all_rw_events=True,
        vertical_lines=[k * SLICE_WIDTH + SLICE_LO for k in range(1, n_slices)],
        detvar_sys_frac_cov_dic_override=None if detvar_override is None else detvar_override(dv, SLICED_VAR, slice_bins),
        show=False, return_p_value_info=True,
    )
    fig = plt.gcf()
    main_ax = next(a for a in fig.axes if a.get_ylabel().startswith("Counts"))
    if detvar_override is not None:
        add_recomb2_panel(fig, slice_bins)
    for k, (lo, hi) in enumerate(zip(SLICE_EDGES[:-1], SLICE_EDGES[1:])):
        main_ax.text(SLICE_LO + (k + 0.5) * SLICE_WIDTH, 0.80, f"{lo}° ≤ α < {hi}°", transform=main_ax.get_xaxis_transform(),
                     ha="center", va="top", fontsize=10, bbox=dict(facecolor="white", alpha=0.8, lw=0))
    # tick labels show the dQ/dx value within each slice
    ticks = [k * SLICE_WIDTH + v for k in range(n_slices) for v in (1.0, 2.0)]
    for a in fig.axes:
        lo_, hi_ = a.get_xlim()
        # only the panels whose x axis is the sliced variable (not the decomposition index axis)
        if SLICE_LO - 0.3 <= lo_ <= SLICE_LO + 0.3 and abs(hi_ - (SLICE_LO + n_slices * SLICE_WIDTH)) < 0.3:
            a.set_xticks(ticks, [f"{(t - SLICE_LO) % SLICE_WIDTH + SLICE_LO:.0f}" for t in ticks])
            if a is not main_ax:  # slice boundaries on the breakdown and DetVar panels too
                for k in range(1, n_slices):
                    a.axvline(SLICE_LO + k * SLICE_WIDTH, color="k", ls="--", lw=0.8)
    fig.text(0.5, 0.92, "α = arccos(|sin θ · cos φ|), θ = wc_cosmict_2_theta, φ = wc_cosmict_2_phi;  "
             f"dQ/dx clipped to [{SLICE_LO}, {SLICE_HI}) in each slice", ha="center", va="bottom", fontsize=11)
    save(fig, name)
    plt.show()
    return info

sliced_info = {}
sliced_info["all cosmict_2 tracks"] = sliced_plot(None, "", "wc_generic_sel__cosmict2_filled_angle_slices", "angle_slices_full_systematics")

In [ ]:
stopping_cut = (pl.col("wc_cosmict_2_particle_type") == 13) & (pl.col("wc_cosmict_2_flag_inside") == 1)
sliced_info["stopping muon-like"] = sliced_plot(stopping_cut, ", particle_type == 13, flag_inside == 1",
                                                "wc_generic_sel__cosmict2_stopping_mu_angle_slices", "angle_slices_full_systematics_stopping_muons")

In [ ]:
from scipy.stats import chi2 as chi2_dist, norm
rows = []
for name, info in sliced_info.items():
    for cov, label in [("nodetvar", "no DetVar"), ("tot", "with DetVar")]:
        c, ndf = info[f"{cov}_chi2"], info["ndf"]
        p = chi2_dist.sf(c, ndf)
        rows.append(dict(selection=name, covariance=label, chi2=round(c, 1), ndf=ndf, chi2_per_ndf=round(c / ndf, 2),
                         p_value=p, sigma=round(norm.isf(p / 2), 2) if p > 0 else np.inf))
pd.DataFrame(rows)

## 6. Recomb2 decorrelated in true muon angle

Section 5 shows the Recomb2 DetVar covariance can't absorb the angle dependence. It moves the stopping-muon peak by the same amount at every angle,
fully correlated, while data is shifted only at large α. As a test, the Recomb2 term is split into independent pieces by the **true** muon angle to the drift direction,
α_true = arccos(|p_x| / |p|) from `wc_truth_muonMomentum_*`, in the same regions 0–30°, 30–50°, 50–70°, 70–90°, plus a separate piece for events with no true muon.
The true angle is the same in the CV and the Recomb2 sample of an event, so the events don't migrate between pieces.
Piece r is built exactly like the standard Recomb2 term, but from only the matched events in region r:
cov_r = d_r d_rᵀ + (bootstrap stat. covariance of those events), with d_r = (Recomb2 − CV) histogram of region-r events. The pieces are added in place of the single Recomb2 term.

**This is not automatically conservative.** In any bin with events from several regions, the variance changes from (Σ d_r)² to Σ d_r²,
which is smaller when the regions shift the same way. So in projections where the angles mix, like the original 1D plot, the Recomb2 uncertainty *shrinks*.
That is checked below on the original 1D plot as well as on the sliced plots.

In [ ]:
from src.systematics import create_detvar_frac_cov_matrices, _bootstrap_hist_cov
from src.plot_helpers import auto_binning
from src.df_helpers import get_vals

TRUE_ALPHA_EDGES = [0, 30, 50, 70, 90]
true_regions = [(-1, "no true μ")] + [(i, f"μ α_true {lo}°–{hi}°") for i, (lo, hi) in enumerate(zip(TRUE_ALPHA_EDGES[:-1], TRUE_ALPHA_EDGES[1:]))]
true_proton_regions = [(-1, "no true p")] + [(i, f"p α_true {lo}°–{hi}°") for i, (lo, hi) in enumerate(zip(TRUE_ALPHA_EDGES[:-1], TRUE_ALPHA_EDGES[1:]))]

def region_from_alpha(alpha, valid):
    region = pl.lit(-1)
    for i, (lo, hi) in enumerate(zip(TRUE_ALPHA_EDGES[:-1], TRUE_ALPHA_EDGES[1:])):
        upper = (alpha <= hi) if i == len(TRUE_ALPHA_EDGES) - 2 else (alpha < hi)
        region = pl.when(valid & (alpha >= lo) & upper).then(pl.lit(i)).otherwise(region)
    return region

def add_true_proton_region(d):
    # leading primary proton direction: costheta = pz/|p|, phi = atan2(py, px) in degrees; sentinels -2 / -999 when there is no proton
    cth = pl.col("wc_true_max_prim_proton_costheta").cast(pl.Float64)
    ph = pl.col("wc_true_max_prim_proton_phi").cast(pl.Float64) * np.pi / 180
    valid = (cth >= -1) & (cth <= 1) & (pl.col("wc_true_max_prim_proton_phi") > -900)
    sinth = (1 - cth.clip(-1, 1) ** 2).sqrt()
    alpha = (sinth * ph.cos()).abs().clip(0, 1).arccos() * 180 / np.pi
    return d.with_columns(region_from_alpha(alpha, valid).alias("true_mu_region"))  # same column name, so recomb2_decorrelated_dic is shared

def add_true_region(d):
    px, py, pz, e = (pl.col(f"wc_truth_muonMomentum_{i}").cast(pl.Float64) for i in range(4))
    pmag = (px ** 2 + py ** 2 + pz ** 2).sqrt()
    alpha = (px.abs() / pmag).clip(0, 1).arccos() * 180 / np.pi
    region = pl.lit(-1)
    for i, (lo, hi) in enumerate(zip(TRUE_ALPHA_EDGES[:-1], TRUE_ALPHA_EDGES[1:])):
        upper = (alpha <= hi) if i == len(TRUE_ALPHA_EDGES) - 2 else (alpha < hi)
        region = pl.when((e > 0) & (pmag > 0) & (alpha >= lo) & upper).then(pl.lit(i)).otherwise(region)
    return d.with_columns(region.alias("true_mu_region"))

def recomb2_decorrelated_dic(dv, var, bins, region_fn=None, regions=None):
    """Standard DetVar fractional covariances, with Recomb2 replaced by independent pieces per true-muon-angle region."""
    region_fn = add_true_region if region_fn is None else region_fn
    regions = true_regions if regions is None else regions
    dv = region_fn(dv)
    dic = create_detvar_frac_cov_matrices(dv, var, bins, use_detvar_bootstrapping=True)
    wcol = detvar_weight_col("Recomb2")
    cv = dv.filter((pl.col("vartype") == "CV") & pl.col(wcol).is_not_null())
    rv = dv.filter(pl.col("vartype") == "Recomb2")
    cv_m = cv.join(rv.select(DETVAR_MATCH_KEYS), on=DETVAR_MATCH_KEYS, how="inner")   # same matching as the standard term
    rv_m = rv.join(cv_m.select(DETVAR_MATCH_KEYS), on=DETVAR_MATCH_KEYS, how="inner")
    cv_counts = np.histogram(get_vals(cv_m, var), weights=get_vals(cv_m, wcol), bins=bins)[0]
    denom = np.outer(cv_counts, cv_counts)
    total_piece = np.zeros_like(denom, dtype=float)
    pieces = {}
    for r, label in regions:
        c, v = cv_m.filter(pl.col("true_mu_region") == r), rv_m.filter(pl.col("true_mu_region") == r)
        cvv, cw, vv, vw = get_vals(c, var), get_vals(c, wcol), get_vals(v, var), get_vals(v, wcol)
        d = np.histogram(vv, weights=vw, bins=bins)[0] - np.histogram(cvv, weights=cw, bins=bins)[0]
        cov = np.outer(d, d)
        if len(cvv):
            cov = cov + _bootstrap_hist_cov(cvv, cw, bins)
        if len(vv):
            cov = cov + _bootstrap_hist_cov(vv, vw, bins)
        frac = np.nan_to_num(np.divide(cov, denom, out=np.zeros_like(cov, dtype=float), where=denom != 0))
        pieces[f"Recomb2 {label}"] = frac
        total_piece += frac
    global last_recomb2_components
    pieces_diff = {}
    for r, label in regions:
        c, v = cv_m.filter(pl.col("true_mu_region") == r), rv_m.filter(pl.col("true_mu_region") == r)
        pieces_diff[label] = (np.histogram(get_vals(v, var), weights=get_vals(v, wcol), bins=bins)[0]
                              - np.histogram(get_vals(c, var), weights=get_vals(c, wcol), bins=bins)[0])
    last_recomb2_components = dict(cv=cv_counts, total_diff=sum(pieces_diff.values()), pieces=pieces_diff)
    full = dic.pop("Recomb2")
    print(f"Recomb2 diagonal error, summed over bins: correlated {np.sqrt(np.diag(full)).sum():.3f}, decorrelated {np.sqrt(np.diag(total_piece)).sum():.3f}")
    dic.update(pieces)
    return dic

In [ ]:
# the original 1D plot (preselection, all events, the automatic binning of plot_many_variables.py), with and without the decorrelation
def original_plot(detvar_override, name, title):
    d, dv = add_true_region(presel_all_df), detvar_all_df
    pred_d, data_d = d.filter(pred_filter), d.filter(pl.col("filetype") == "data")
    var = "wc_cosmict_2_dQ_dx_end"
    bins, display_bins = auto_binning(np.concatenate([get_vals(pred_d, var), get_vals(data_d, var)]))[:2]  # the bins make_histogram_plot picks
    info = make_histogram_plot(
        pred_sel_df=pred_d, data_sel_df=data_d, include_overflow=False, include_underflow=False, log_y=True, include_legend=False,
        var=var, title=title, selname="wc_generic_sel",
        include_ratio=True, include_decomposition=True, include_sys_breakdown_panels=True, include_detvar_variation_panels=True,
        use_rw_systematics=True, use_detvar_systematics=True, detvar_df=dv, weights_df=weights_df, require_all_rw_events=True,
        detvar_sys_frac_cov_dic_override=None if detvar_override is None else detvar_override(dv, var, bins),
        show=False, return_p_value_info=True,
    )
    if detvar_override is not None:
        add_recomb2_panel(plt.gcf(), display_bins)
    save(plt.gcf(), name)
    plt.show()
    return info

decor_info = {}
decor_info[("1D preselection", "standard")] = original_plot(None, "original_1d_standard", "Preselection")
decor_info[("1D preselection", "Recomb2 decorrelated in μ angle")] = original_plot(recomb2_decorrelated_dic, "original_1d_recomb2_decorrelated",
                                                                        "Preselection, Recomb2 decorrelated in true μ angle")

In [ ]:
decor_info[("angle slices, all tracks", "standard")] = sliced_info["all cosmict_2 tracks"]
decor_info[("angle slices, stopping μ-like", "standard")] = sliced_info["stopping muon-like"]
decor_info[("angle slices, all tracks", "Recomb2 decorrelated in μ angle")] = sliced_plot(
    None, ", Recomb2 decorrelated in true μ angle", "wc_generic_sel__cosmict2_filled_angle_slices",
    "angle_slices_recomb2_decorrelated", detvar_override=lambda dv, var, bins: recomb2_decorrelated_dic(dv, var, bins))
decor_info[("angle slices, stopping μ-like", "Recomb2 decorrelated in μ angle")] = sliced_plot(
    stopping_cut, ", particle_type == 13, flag_inside == 1, Recomb2 decorrelated in true μ angle", "wc_generic_sel__cosmict2_stopping_mu_angle_slices",
    "angle_slices_recomb2_decorrelated_stopping_muons", detvar_override=lambda dv, var, bins: recomb2_decorrelated_dic(dv, var, bins))

### Decorrelated in true leading-proton angle instead

Same construction, with the regions defined by the angle of the leading true primary proton (`wc_true_max_prim_proton_costheta`, `_phi`)
to the drift direction, plus a piece for events with no true primary proton.

In [ ]:
proton_override = lambda dv, var, bins: recomb2_decorrelated_dic(dv, var, bins, region_fn=add_true_proton_region, regions=true_proton_regions)
decor_info[("1D preselection", "Recomb2 decorrelated in p angle")] = original_plot(
    proton_override, "original_1d_recomb2_decorrelated_proton", "Preselection, Recomb2 decorrelated in true leading-p angle")
decor_info[("angle slices, all tracks", "Recomb2 decorrelated in p angle")] = sliced_plot(
    None, ", Recomb2 decorrelated in true leading-p angle", "wc_generic_sel__cosmict2_filled_angle_slices",
    "angle_slices_recomb2_decorrelated_proton", detvar_override=proton_override)
decor_info[("angle slices, stopping μ-like", "Recomb2 decorrelated in p angle")] = sliced_plot(
    stopping_cut, ", particle_type == 13, flag_inside == 1, Recomb2 decorrelated in true leading-p angle",
    "wc_generic_sel__cosmict2_stopping_mu_angle_slices", "angle_slices_recomb2_decorrelated_proton_stopping_muons", detvar_override=proton_override)

In [ ]:
rows = []
for (plot, treatment), info in decor_info.items():
    for cov, label in [("nodetvar", "no DetVar"), ("tot", "with DetVar")]:
        if cov == "nodetvar" and treatment != "standard":
            continue  # identical to the standard one
        c, ndf = info[f"{cov}_chi2"], info["ndf"]
        p = chi2_dist.sf(c, ndf)
        rows.append(dict(plot=plot, covariance=label if cov == "nodetvar" else f"with DetVar, {treatment}",
                         chi2=round(c, 1), ndf=ndf, sigma=round(norm.isf(p / 2), 2) if p > 0 else np.inf))
decor_table = pd.DataFrame(rows).sort_values(["plot", "covariance"]).reset_index(drop=True)
decor_table

## Findings (run of 2026-11-17, presel_df_train_vars from 2026-11-04)

- **The distribution has two peaks.** The MIP peak is near 1.0–1.1. A second peak near 1.8–1.9 comes from the end of stopping muon tracks.
  In data the second peak is shifted up. For stopping muon-like tracks (particle_type 13, flag_inside 1) the weighted median in [1.4, 2.8] is
  **1.90 in data vs 1.83 in the prediction (+4%)**. That produces the dip at 1.5–1.8 and the excess at 2.0–2.4 (excess-window fraction data/pred ≈ 1.5).
- **It doesn't change with time.** The data/prediction excess ratio is 1.46–1.59 in every run period, and the data peak position is 1.89–1.91 in every period.
- **It is specific to stopping muon tracks.**
  - Muon-like tracks: ratio 1.68. Proton-like tracks: 1.09.
  - Track end inside the detector: 1.58. Exiting tracks: 1.14.
  - `dQ_dx_front` on the same tracks agrees.
- **It depends on track angle.** By angle to the drift direction, the ratio is about 1.07 for tracks within 30° of the drift direction,
  1.28 at 30–45°, and about 1.65 for tracks at 60–90° (nearly parallel to the wire planes). Forward-going tracks (< 30° to the beam) show it most (1.86).
  The azimuth has no clear dependence.
- **EXT vs overlay cosmics.** These are real-data cosmic tracks in both cases, and they show a shift in the same direction:
  the overlay cosmic-dominated median is 1.79 vs 1.85 for EXT, and the area-normalized ratio has the same dip and excess pattern.
  That part cannot come from the simulated physics of the track. It points to the overlay processing (calibration) giving lower end-of-track dQ/dx than the data processing.
  One caveat is that the two samples may contain a different mix of cosmic topologies.
- **DetVar.** The Recomb2 variation reproduces the data/prediction shape: dip −28% vs data −22%, excess +71% vs data +53%, median +0.018 vs data +0.015 (core range).
  It also covers it, so the χ² with DetVar is 62.6/22 rather than ~570/22. The other variations are small apart from WireModThetaYZ (+17% in the excess window).
  On the low side of the MIP peak (0.6–0.9) data is above the prediction, while Recomb2 goes the other way, so Recomb2 isn't the full story.

**Interpretation.** The likely causes are the recombination modeling at high dE/dx, which depends on angle (Recomb2 matches the shape and the angle dependence points there),
and an overlay vs data calibration difference for end-of-track dQ/dx (EXT vs overlay cosmics).
For the BDT, this variable carries a known ~4% scale difference at the stopping peak. Recomb2 covers it with a large error, so it is not a problem for the
uncertainty treatment, but the BDT could pick up a data/MC difference here.

**Angle slices with full systematics (section 5).** Splitting `dQ_dx_end` into the four α slices (48 bins) makes the disagreement much more significant:

| selection | no DetVar χ²/ndf | with DetVar χ²/ndf | σ with DetVar |
|---|---|---|---|
| all cosmic-tagger-2 tracks | 774.1/48 | 171.1/48 | 8.0 |
| stopping muon-like | 828.9/48 | 103.2/48 | 4.5 |

For comparison, the 1D variable without the angle split gives 62.6/22 (4.4σ) with DetVar.
The DetVar uncertainty (mostly Recomb2) covers a dQ/dx shift at every angle at once, but data is shifted only at large α, so the angle dependence isn't absorbed.
Part of the all-tracks χ² is also the angular rate mismatch (data has relatively more tracks along the drift: ~25% more in the 0°–30° slice); the stopping muon-like selection has less of it.

**Recomb2 decorrelated in true muon angle (section 6).** With DetVar:

| plot | standard Recomb2 | decorrelated Recomb2 |
|---|---|---|
| original 1D (preselection) | 62.6/22 (4.4σ) | 56.6/22 (4.0σ) |
| angle slices, all tracks | 171.1/48 (8.0σ) | 129.3/48 (6.0σ) |
| angle slices, stopping μ-like | 103.2/48 (4.5σ) | 69.4/48 (2.3σ) |

The standard 1D value reproduces the plot_many_variables result, which validates the setup. Decorrelation removes about half of the excess χ² in the stopping-muon slices,
and still slightly improves the 1D χ², even though it shrinks the summed Recomb2 diagonal error (4.38 → 2.92 in the 1D binning).
The all-tracks slices stay at 6σ, mostly from the angular rate mismatch (excess of tracks along the drift direction), which Recomb2 doesn't touch.

**Decorrelating in true leading-proton angle instead (section 6).** χ²/ndf with DetVar:

| plot | standard | decorrelated, μ angle | decorrelated, leading-p angle |
|---|---|---|---|
| original 1D | 62.6/22 (4.4σ) | 56.6/22 (4.0σ) | 63.2/22 (4.5σ) |
| angle slices, all tracks | 171.1/48 (8.0σ) | 129.3/48 (6.0σ) | 164.1/48 (7.7σ) |
| angle slices, stopping μ-like | 103.2/48 (4.5σ) | 69.4/48 (2.3σ) | 99.2/48 (4.3σ) |

The proton-angle split shrinks the Recomb2 errors at least as much as the muon split (summed diagonal 7.4 vs 9.8 in the all-tracks slices), but leaves χ² almost unchanged.
So the improvement with the muon split is not just a generic effect of extra freedom: it works because the muon angle is the angle of the tagged track whose end dQ/dx is mismodelled.